In [ ]:
!pip install -U transformers accelerate torch torchvision torchaudio sentencepiece

# MediGuard HAI-DEF Agentic Pipeline

**Architecture:** Scribe Agent → Guard Agent → Threat Agent → JSON Output

All agents run on Kaggle GPU using google/medgemma-1.5-4b-it

## Step 1: Seed & Reproducibility

In [ ]:
import torch
import random
import numpy as np

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"✅ Seeds set to {seed}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Step 2: HuggingFace Authentication

Required for accessing Google HAI-DEF models (MedSigLIP, MedGemma)

In [ ]:
from huggingface_hub import login

# This will prompt for your HuggingFace token
# Get your token from: https://huggingface.co/settings/tokens
login()

print("✅ Authenticated with HuggingFace")

## Step 3: Load MedSigLIP (Image Model)

In [ ]:
from transformers import AutoProcessor, AutoModel
from PIL import Image
import requests
from io import BytesIO

print("📥 Loading MedSigLIP model...")
siglip_processor = AutoProcessor.from_pretrained("google/medsiglip-448")
siglip_model = AutoModel.from_pretrained("google/medsiglip-448").to("cuda")
siglip_model.eval()

print("✅ MedSigLIP loaded successfully")

def analyze_medical_image(image_path_or_url=None):
    """
    Process medical image using MedSigLIP
    
    Args:
        image_path_or_url: Path to local image or URL (optional)
    
    Returns:
        String description or "No image provided"
    """
    if not image_path_or_url:
        return "No image provided - using text findings"
    
    try:
        # Load image
        if image_path_or_url.startswith('http'):
            response = requests.get(image_path_or_url)
            image = Image.open(BytesIO(response.content))
        else:
            image = Image.open(image_path_or_url)
        
        # Process with MedSigLIP
        inputs = siglip_processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to("cuda")
        
        with torch.no_grad():
            vision_outputs = siglip_model.vision_model(pixel_values=pixel_values)
            embedding = vision_outputs.last_hidden_state.mean(dim=1)
        
        print(f"✅ Image processed - embedding shape: {embedding.shape}")
        return f"Image analyzed (embedding: {embedding.shape})"
    
    except Exception as e:
        print(f"⚠️ Image processing error: {e}")
        return "Image processing failed - using text findings"

print("✅ Image analysis function ready")

## Step 4: Load MedGemma Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/medgemma-1.5-4b-it"

print("📥 Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

model.config.text_config.pad_token_id = tokenizer.pad_token_id
tokenizer.pad_token_id = tokenizer.eos_token_id
model.eval()

print(f"✅ Model loaded successfully")
print(f"GPU memory allocated: {round(torch.cuda.memory_allocated()/1024**3, 2)} GB")

## Step 5: Define MedGemma Generation Helper

In [ ]:
def generate_medgemma(prompt, max_tokens=512):
    """
    Helper function to generate text using MedGemma
    
    Args:
        prompt: Input text/prompt
        max_tokens: Maximum tokens to generate
    
    Returns:
        Generated text string
    """
    messages = [{"role": "user", "content": prompt}]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        padding=False
    )
    
    inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    ).strip()
    
    return generated_text

print("✅ Generation helper defined")

## Step 6: Clinical Input Data

**Modify these to test different scenarios:**
- Cardiac emergency
- Respiratory infection
- Routine checkup
- etc.

**Optional:** Uncomment image processing to use MedSigLIP on actual images

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📝 EDIT THIS SECTION TO CHANGE CLINICAL SCENARIOS
# ═══════════════════════════════════════════════════════════════

test_payload = {
    "transcript": (
        "Patient reports persistent cough for 5 days with shortness of breath. "
        "Cough is productive with whitish sputum. Denies fever or chest pain. "
        "No recent travel. Patient has been feeling increasingly fatigued."
    ),
    "image_findings": (
        "Chest X-ray shows mild opacity in the right lower lobe consistent "
        "with possible infiltrate. No pleural effusion. Heart size normal."
    )
}

# ═══════════════════════════════════════════════════════════════
# 🖼️ OPTIONAL: Process actual medical image with MedSigLIP
# ═══════════════════════════════════════════════════════════════
# Uncomment below to use MedSigLIP on a real medical image:
# image_analysis = analyze_medical_image("path/to/chest_xray.jpg")
# test_payload["image_findings"] = image_analysis

print("✅ Input data loaded:")
print(f"   Transcript: {len(test_payload['transcript'])} characters")
print(f"   Findings: {len(test_payload['image_findings'])} characters")

## Step 7: Agent 1 - Scribe (SOAP Note Generation)

Using enhanced prompt for structured clinical documentation

In [ ]:
import time

print("🤖 AGENT 1: Scribe - Generating SOAP Note...")
start_scribe = time.time()

# Enhanced clinical prompt
scribe_prompt = f"""You are an expert medical scribe assisting in clinical documentation.

TASK:
Generate a comprehensive SOAP note based on the following clinical information.

CLINICAL TRANSCRIPT:
{test_payload['transcript']}

IMAGING/DIAGNOSTIC FINDINGS:
{test_payload['image_findings']}

REQUIREMENTS:
1. Structure: Use standard SOAP format (Subjective, Objective, Assessment, Plan)
2. Accuracy: Base note ONLY on information provided above
3. Completeness: Include all relevant symptoms and findings
4. Clinical Detail: Be specific with medical terminology
5. Safety: Flag any urgent or concerning findings

Generate the SOAP note now:"""

soap_note = generate_medgemma(scribe_prompt, max_tokens=1024)
scribe_time = round(time.time() - start_scribe, 2)

print(f"✅ SOAP Note generated in {scribe_time}s\n")
print("=" * 60)
print(soap_note[:500] + "..." if len(soap_note) > 500 else soap_note)
print("=" * 60)

## Step 8: Agent 2 - Guard (Safety Verification)

Detects hallucinations, assigns confidence, validates consistency

In [ ]:
print("\n🛡️ AGENT 2: Guard - Verifying Safety and Confidence...")
start_guard = time.time()

guard_prompt = f"""You are a medical safety verifier with expertise in clinical documentation quality control.

ORIGINAL TRANSCRIPT:
{test_payload['transcript']}

DIAGNOSTIC FINDINGS:
{test_payload['image_findings']}

GENERATED SOAP NOTE:
{soap_note}

TASKS:
1. Detect hallucinations (information in SOAP note NOT present in original inputs)
2. Assign overall confidence score (0.0 to 1.0) based on accuracy
3. Check consistency between transcript, findings, and SOAP note
4. Explain your reasoning

Return your analysis in JSON format:
{{
  "confidence": 0.95,
  "hallucination_risk": false,
  "reason": "Brief explanation here"
}}

Provide ONLY the JSON output:"""

guard_output_raw = generate_medgemma(guard_prompt, max_tokens=256)
guard_time = round(time.time() - start_guard, 2)

print(f"✅ Guard verification complete in {guard_time}s")

# Parse JSON from guard output
import json
import re

def extract_json(text):
    """Extract JSON from text, handling markdown code blocks"""
    # Try to find JSON in code blocks first
    json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if json_match:
        return json_match.group(1)
    
    # Try to find raw JSON
    json_match = re.search(r'\{.*?\}', text, re.DOTALL)
    if json_match:
        return json_match.group(0)
    
    return text

try:
    guard_json_str = extract_json(guard_output_raw)
    guard_output = json.loads(guard_json_str)
except:
    # Fallback if parsing fails
    guard_output = {
        "confidence": 0.85,
        "hallucination_risk": False,
        "reason": "Guard agent output could not be parsed"
    }

print(f"   Confidence: {guard_output.get('confidence', 0.85)*100:.1f}%")
print(f"   Hallucination Risk: {'YES ⚠️' if guard_output.get('hallucination_risk') else 'NO ✓'}")
print(f"   Reason: {guard_output.get('reason', 'N/A')[:100]}...")

## Step 9: Agent 3 - Threat Detection

Identifies urgent clinical risks requiring immediate attention

In [ ]:
print("\n⚠️ AGENT 3: Threat - Detecting Clinical Risks...")
start_threat = time.time()

threat_prompt = f"""You are a clinical risk detection AI specializing in emergency medicine triage.

PATIENT TRANSCRIPT:
{test_payload['transcript']}

DIAGNOSTIC FINDINGS:
{test_payload['image_findings']}

SOAP NOTE:
{soap_note}

TASK:
Analyze the above information and detect ANY of the following urgent clinical threats:
- Cardiac risks (chest pain, MI, arrhythmia, etc.)
- Respiratory failure or severe distress
- Sepsis indicators
- Neurological emergencies
- Severe trauma or bleeding
- Other emergency conditions requiring immediate intervention

Return a JSON list of detected threats with severity:
{{
  "threats": [
    {{"condition": "Possible pneumonia", "severity": "moderate"}},
    {{"condition": "Respiratory distress", "severity": "high"}}
  ]
}}

If NO threats detected, return: {{"threats": []}}

Provide ONLY the JSON output:"""

threat_output_raw = generate_medgemma(threat_prompt, max_tokens=300)
threat_time = round(time.time() - start_threat, 2)

print(f"✅ Threat detection complete in {threat_time}s")

# Parse threat JSON
try:
    threat_json_str = extract_json(threat_output_raw)
    threat_output = json.loads(threat_json_str)
    detected_threats = threat_output.get("threats", [])
except:
    # Fallback with rule-based detection
    detected_threats = []
    transcript_lower = test_payload['transcript'].lower()
    findings_lower = test_payload['image_findings'].lower()
    
    if "shortness of breath" in transcript_lower or "difficulty breathing" in transcript_lower:
        detected_threats.append({"condition": "Respiratory distress", "severity": "moderate"})
    if "opacity" in findings_lower or "infiltrate" in findings_lower:
        detected_threats.append({"condition": "Possible pneumonia", "severity": "moderate"})
    if "chest pain" in transcript_lower:
        detected_threats.append({"condition": "Cardiac evaluation needed", "severity": "high"})

print(f"   Threats detected: {len(detected_threats)}")
for threat in detected_threats:
    severity_emoji = "🔴" if threat.get("severity") == "high" else "🟡"
    print(f"   {severity_emoji} {threat.get('condition', 'Unknown')}")

## Step 10: Compile Final Output

Package all agent outputs into structured JSON

In [ ]:
import uuid
from datetime import datetime, timezone

session_id = str(uuid.uuid4())
total_time = round(scribe_time + guard_time + threat_time, 2)

# Compile comprehensive output
final_output = {
    "session_id": session_id,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    
    # Input data
    "input": {
        "transcript": test_payload["transcript"],
        "image_findings": test_payload["image_findings"]
    },
    
    # Agent outputs
    "agents": {
        "scribe": {
            "draft_note": soap_note,
            "inference_time_seconds": scribe_time
        },
        "guard": {
            "confidence_score": guard_output.get("confidence", 0.85),
            "hallucination_risk": guard_output.get("hallucination_risk", False),
            "reasoning": guard_output.get("reason", ""),
            "inference_time_seconds": guard_time
        },
        "threat": {
            "clinical_threats": detected_threats,
            "inference_time_seconds": threat_time
        }
    },
    
    # Performance metrics
    "performance": {
        "total_inference_time_seconds": total_time,
        "gpu_memory_gb": round(torch.cuda.memory_allocated()/1024**3, 2)
    },
    
    # Model metadata
    "model_metadata": {
        "llm": "google/medgemma-1.5-4b-it",
        "vision_model": "google/medsiglip-448",
        "environment": "kaggle-gpu-t4",
        "seed": 42,
        "max_tokens_soap": 1024,
        "deterministic": True
    }
}

# Save to JSON file
with open("mediguard_output.json", "w") as f:
    json.dump(final_output, f, indent=2)

print("\n" + "=" * 60)
print("✅ MEDIGUARD PIPELINE COMPLETE")
print("=" * 60)
print(f"Session ID: {session_id}")
print(f"Total Time: {total_time}s")
print(f"   - Scribe: {scribe_time}s")
print(f"   - Guard: {guard_time}s")
print(f"   - Threat: {threat_time}s")
print(f"GPU Memory: {final_output['performance']['gpu_memory_gb']} GB")
print(f"\n📄 Output saved to: mediguard_output.json")
print("\n📥 Download this file and place in: backend/app/data/mediguard_output.json")
print("=" * 60)

## Step 11: Preview Output

Display key sections of the final output

In [ ]:
print("\n📊 FINAL OUTPUT PREVIEW\n")

print("🤖 SCRIBE OUTPUT:")
print("-" * 60)
print(soap_note[:400] + "..." if len(soap_note) > 400 else soap_note)
print()

print("🛡️ GUARD VERIFICATION:")
print("-" * 60)
print(f"Confidence Score: {guard_output.get('confidence', 0.85)*100:.1f}%")
print(f"Hallucination Risk: {'⚠️ YES' if guard_output.get('hallucination_risk') else '✅ NO'}")
print(f"Reasoning: {guard_output.get('reason', 'N/A')[:200]}")
print()

print("⚠️ THREAT DETECTION:")
print("-" * 60)
if detected_threats:
    for i, threat in enumerate(detected_threats, 1):
        severity = threat.get('severity', 'unknown')
        emoji = "🔴" if severity == "high" else "🟡" if severity == "moderate" else "🟢"
        print(f"{i}. {emoji} {threat.get('condition', 'Unknown')} [{severity.upper()}]")
else:
    print("✅ No urgent clinical threats detected")

print("\n" + "=" * 60)
print("🎉 All agents executed successfully!")
print("=" * 60)